# Interactive HypCBM Interventions

Explore ImageNet predictions of a trained Hyperbolic Concept Bottleneck Model (HypCBM), visualize the concept evidence behind each decision, and perform causal interventions to see how suppressing or boosting concepts alters the outcome.

## 1. Environment Setup and Global Configuration
Import all required libraries, point to the trained artifact and concept bank, and define plotting / device defaults. Update the paths below if you retrained the model elsewhere.

In [ ]:
import pickle
from pathlib import Path
from types import SimpleNamespace
from typing import Dict, Iterable, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
from torchvision import datasets

from concepts import ConceptBank
from models.entailment_cbm import entailment_cone_feature_matrix
from utils.intervention_utils import apply_interventions_to_features
from models import PosthocLinearCBM, get_model

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["axes.grid"] = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BACKBONE_NAME = "hycoclip"
HYC_CONFIG = "hycoclip/configs/train_hycoclip_vit_b.py"
HYC_CHECKPOINT = "hycoclip/hycoclip_vit_b.pth"
DEFAULT_ETA = 1.3  # override to sweep different propagation strengths

DATASET_NAME = "sun397"  # toggle between "imagenet" and "sun397" as needed

DATASET_CONFIGS = {
    "imagenet": {
        "artifact_path": Path("outputs_imagenet/entailment_cbm_imagenet_hycoclip_1frac.pkl"),
        "concept_bank": Path("outputs_imagenet/multimodal_concept_hycoclip_imagenet_recurse:1_minnorm:0.27.pkl"),
        "data_root": Path("/path/to/imagenet/torchvision_ImageFolder"),
        "split": "val",
        "class_index_json": Path("imagenet_class_index.json"),
        "max_concept_bars": 15,
        "max_class_bars": 5,
        "split_seed": 42,
    },
    "sun397": {
        "artifact_path": Path("outputs_sun397/entailment_cbm_sun397_hycoclip_1frac.pkl"),
        "concept_bank": Path("outputs_sun397/multimodal_concept_hycoclip_sun397_recurse:1_minnorm:0.27.pkl"),
        "data_root": Path("data/sun397/SUN_dataset/data"),
        "split": "full",
        "max_concept_bars": 15,
        "max_class_bars": 5,
        "val_fraction": 0.1,
        "split_seed": 0,
    },
}

if DATASET_NAME not in DATASET_CONFIGS:
    raise ValueError(f"Unsupported dataset '{DATASET_NAME}'. Choose from: {list(DATASET_CONFIGS)}")

active_cfg = DATASET_CONFIGS[DATASET_NAME]
ARTIFACT_PATH = Path(active_cfg["artifact_path"]).expanduser()
FALLBACK_CONCEPT_BANK = Path(active_cfg["concept_bank"]).expanduser()
DATA_ROOT = Path(active_cfg["data_root"]).expanduser()
DATASET_SPLIT = active_cfg.get("split", "val")
CLASS_INDEX_JSON = active_cfg.get("class_index_json")
if CLASS_INDEX_JSON is not None:
    CLASS_INDEX_JSON = Path(CLASS_INDEX_JSON).expanduser()
DEFAULT_MAX_CONCEPT_BARS = int(active_cfg.get("max_concept_bars", 15))
DEFAULT_MAX_CLASS_BARS = int(active_cfg.get("max_class_bars", 5))
DATASET_VAL_FRACTION = float(active_cfg.get("val_fraction", 0.1))
DATASET_SPLIT_SEED = int(active_cfg.get("split_seed", 0))
MAX_CONCEPT_BARS = DEFAULT_MAX_CONCEPT_BARS
MAX_CLASS_BARS = DEFAULT_MAX_CLASS_BARS

## 2. Load Entailment CBM Artifact and HyCoCLIP Backbone
Load the trained CBM artifact, the associated concept bank, the HyCoCLIP backbone (for curvature + embeddings), and the post-hoc classifier head.

In [ ]:
if not ARTIFACT_PATH.exists():
    raise FileNotFoundError(f"Artifact not found at {ARTIFACT_PATH}. Update ARTIFACT_PATH above.")

with ARTIFACT_PATH.open("rb") as handle:
    artifact: Dict = pickle.load(handle)
print(artifact['run_info'])
concept_bank_path = Path(artifact.get("concept_bank", FALLBACK_CONCEPT_BANK))
if not concept_bank_path.exists():
    raise FileNotFoundError(f"Concept bank not found at {concept_bank_path}.")

artifact_chunk_size = int(artifact.get("chunk_size", 64))
artifact_eta = float(artifact.get("entail_eta", 0.9))
ANALYSIS_ETA = artifact_eta if DEFAULT_ETA is None else DEFAULT_ETA
artifact_weights = torch.as_tensor(np.asarray(artifact["weights"], dtype=np.float32))
artifact_bias = torch.as_tensor(np.asarray(artifact["bias"], dtype=np.float32))

print({"dataset": artifact.get("dataset"), "eta": artifact_eta, "chunk_size": artifact_chunk_size})

device_obj = torch.device(DEVICE)
backbone_args = SimpleNamespace(
    device=DEVICE,
    backbone_name=BACKBONE_NAME,
    hycoclip_config=HYC_CONFIG,
    hycoclip_checkpoint=HYC_CHECKPOINT,
    out_dir="./data",
)
backbone, preprocess = get_model(backbone_args, backbone_name=BACKBONE_NAME)
backbone = backbone.to(device_obj).eval()

concept_data = pickle.load(concept_bank_path.open("rb"))
concept_bank = ConceptBank(concept_data, DEVICE)
concept_tensor = concept_bank.vectors.detach().clone().to(device_obj)
concept_names = concept_bank.concept_names

concept_lookup = {name.lower(): idx for idx, name in enumerate(concept_names)}

posthoc_layer = PosthocLinearCBM(
    concept_bank,
    backbone_name=artifact.get("backbone", BACKBONE_NAME),
    idx_to_class={i: str(i) for i in range(artifact_weights.shape[0])},
    n_classes=artifact_weights.shape[0],
)
posthoc_layer = posthoc_layer.to(device_obj).eval()
posthoc_layer.set_weights(artifact_weights, artifact_bias)

print(f"Loaded {len(concept_names)} concepts and {artifact_weights.shape[0]} classes.")


## 3. Prepare Evaluation Dataset
Instantiate the selected dataset twice: once with the HyCoCLIP preprocessing pipeline (for embeddings) and once with raw PIL images for visualization. Update the DATASET_CONFIGS block above if your data trees live elsewhere.

In [ ]:
# Optional manual override; set either value to an int to diverge from the dataset defaults.
MANUAL_MAX_CONCEPT_BARS = None
MANUAL_MAX_CLASS_BARS = None

if MANUAL_MAX_CONCEPT_BARS is not None:
    MAX_CONCEPT_BARS = int(MANUAL_MAX_CONCEPT_BARS)
if MANUAL_MAX_CLASS_BARS is not None:
    MAX_CLASS_BARS = int(MANUAL_MAX_CLASS_BARS)

print(f"Plotting top {MAX_CONCEPT_BARS} concepts and {MAX_CLASS_BARS} classes per example.")

In [ ]:
split_root = DATA_ROOT if DATASET_NAME == "sun397" else DATA_ROOT / DATASET_SPLIT
if not split_root.exists():
    raise FileNotFoundError(
        f"Dataset split '{DATASET_SPLIT}' not found at {split_root}. Update DATASET_CONFIGS before running."
    )

if DATASET_NAME == "imagenet":
    eval_dataset = datasets.ImageFolder(root=str(split_root), transform=preprocess)
    raw_dataset = datasets.ImageFolder(root=str(split_root), transform=None)
    class_names = eval_dataset.classes

    import json

    if CLASS_INDEX_JSON is not None:
        try:
            with CLASS_INDEX_JSON.open("r") as f:
                imagenet_class_index = json.load(f)
            synset_to_class = {v[0]: v[1] for v in imagenet_class_index.values()}
            class_names = [synset_to_class.get(c, c) for c in eval_dataset.classes]
            print("Updated class_names to human-readable labels.")
        except FileNotFoundError:
            print(f"{CLASS_INDEX_JSON} not found. Using synset IDs.")
elif DATASET_NAME == "sun397":
    full_dataset = datasets.SUN397(root=str(split_root), transform=preprocess, download=False)
    val_fraction = max(min(DATASET_VAL_FRACTION, 0.99), 1e-4)
    split_seed = DATASET_SPLIT_SEED
    val_size = max(1, int(len(full_dataset) * val_fraction))
    train_size = len(full_dataset) - val_size
    generator = torch.Generator().manual_seed(split_seed)
    _, val_subset = torch.utils.data.random_split(full_dataset, [train_size, val_size], generator=generator)

    eval_dataset = val_subset
    base_raw = datasets.SUN397(root=str(split_root), transform=None, download=False)
    subset_indices = getattr(val_subset, "indices", list(range(len(val_subset))))
    raw_dataset = torch.utils.data.Subset(base_raw, subset_indices)
    class_names = full_dataset.classes
else:
    raise ValueError(f"Unsupported dataset '{DATASET_NAME}'.")

print(
    f"Loaded {DATASET_NAME} split with {len(eval_dataset)} samples (val_fraction={val_fraction:.2f} if applicable)."
    if DATASET_NAME == "sun397"
    else f"Loaded {DATASET_NAME} split with {len(eval_dataset)} samples."
 )

def fetch_sample(image_index: int) -> Tuple[torch.Tensor, torch.Tensor, int]:
    tensor_img, label = eval_dataset[image_index]
    raw_img, _ = raw_dataset[image_index]
    return tensor_img.unsqueeze(0).to(device_obj), raw_img, label

## 4. Utility: encode_batch_images
Route images through HyCoCLIP with the correct projection flags to obtain the hyperbolic embedding $z$ as a NumPy array.

In [ ]:
def encode_batch_images(images: torch.Tensor) -> np.ndarray:
    name = BACKBONE_NAME.lower()
    if "hycoclip" in name:
        return backbone.encode_image(images, project=True).detach().cpu().numpy().astype(np.float64)
    elif "clip" in name:
        return backbone.encode_image(images, project=True).detach().cpu().numpy().astype(np.float32)
    elif "meru" in name:
        return backbone.encode_image(images, project=True).detach().cpu().numpy().astype(np.float32)
    return backbone(images).detach().cpu().numpy().astype(np.float64)


def compute_features_and_probs(image_index: int) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    batch_tensor, raw_img, label = fetch_sample(image_index)
    with torch.no_grad():
        embedding = encode_batch_images(batch_tensor)
    features = entailment_cone_feature_matrix(
        backbone,
        embedding.copy(),
        concept_tensor,
        chunk_size=artifact_chunk_size,
        device=device_obj,
        eta=ANALYSIS_ETA,
    )
    feature_tensor = torch.from_numpy(features).to(device_obj)
    with torch.no_grad():
        logits = posthoc_layer.forward_projs(feature_tensor)
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]
    return raw_img, features[0].copy(), probs.copy(), np.asarray(label)



## 5. Helper: analyze_image
Display the raw ImageNet image, report the top concept activations and class probabilities, and print textual diagnostics for a selected index.

In [ ]:
def analyze_image(image_index: int) -> None:
    raw_img, concept_scores, class_probs, label = compute_features_and_probs(image_index)

    pred_idx = int(class_probs.argmax())

    # Retrieve classifier weights for the predicted class
    if hasattr(posthoc_layer, "classifier") and hasattr(posthoc_layer.classifier, "weight"):
        pred_weights = posthoc_layer.classifier.weight[pred_idx].detach().cpu().numpy()
    elif hasattr(posthoc_layer, "classifier_weights"):
        pred_weights = posthoc_layer.classifier_weights()[pred_idx].detach().cpu().numpy()
    else:
        raise AttributeError("PosthocLinearCBM does not expose classifier weights")

    # Calculate contributing scores: weight * activation
    contributions = pred_weights * concept_scores
    
    # Sort and pick top K
    top_concepts = contributions.argsort()[-MAX_CONCEPT_BARS:][::-1]
    top_classes = class_probs.argsort()[-MAX_CLASS_BARS:][::-1]

    # --- Plotting ---
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].imshow(raw_img)
    axes[0].axis("off")
    axes[0].set_title(f"Sample #{image_index}\nTrue: {class_names[label]}")
    # no grid
    axes[1].grid(False)
    axes[1].barh(
        [concept_names[i] for i in top_concepts][::-1],
        contributions[top_concepts][::-1],
        color="tab:blue",
    )
    axes[1].set_title(f"Top concept contributions (for {class_names[pred_idx]})")
    axes[1].set_xlabel("weight × activation")

    axes[2].barh(
        [class_names[i] for i in top_classes][::-1],
        class_probs[top_classes][::-1],
        color="tab:green",
    )
    axes[2].set_title("Top class probabilities")
    axes[2].set_xlabel("probability")
    plt.tight_layout()
    plt.show()

    # --- Printed Output for Intervention ---
    print(f"Predicted class: {class_names[pred_idx]} (p={class_probs[pred_idx]:.3f})")
    print(f"True class:      {class_names[label]}")
    print("\nTop Contributing Concepts (Use 'Index' to intervene):")
    print("-" * 65)
    print(f"{'Index':<8} | {'Concept Name':<35} | {'Contribution':<12}")
    print("-" * 65)
    
    for i in top_concepts:
        # Only show concepts that actively contributed to the prediction (score > 0)
        if contributions[i] > 0:
            print(f"{i:<8d} | {concept_names[i]:<35} | {contributions[i]:+.4f}")
    print("-" * 65)

In [ ]:
def perform_intervention(
    image_index: int,
    interventions: Dict[str, float],
    top_k_classes: int = 5,
    top_k_concepts: int = 5 # Number of top concepts to show context
) -> None:
    """Apply concept interventions and visualize the downstream class shift."""

    if not interventions:
        raise ValueError("Provide at least one concept -> value entry.")

    raw_img, base_concepts, base_probs, label = compute_features_and_probs(image_index)

    intervention_list = []
    # We will track names to plot
    plot_concept_indices = set()
    
    for concept_name, target_value in interventions.items():
        idx = concept_lookup.get(concept_name.lower())
        if idx is None:
            print(f"[warn] Concept '{concept_name}' not found; skipping.")
            continue
        intervention_list.append({"idx": idx, "value": float(target_value)})
        plot_concept_indices.add(idx)

    if not intervention_list:
        print("No valid interventions resolved; aborting.")
        return

    # Apply Geometric Intervention (Propagation)
    # Note: ensure eta here matches your calibrated text-text eta (e.g. 1.3)
    intervened = apply_interventions_to_features(
        base_concepts[None, :],
        intervention_list,
        concept_tensor,
        model=backbone
    )[0]

    ZERO_EPS = 1e-6
    # Find concepts that were ON before but are OFF now
    propagated_mask = (np.abs(intervened) <= ZERO_EPS) & (np.abs(base_concepts) > ZERO_EPS)
    propagated_indices = np.where(propagated_mask)[0]
    
    # Add these 'casualties' to the plot list so we can see them vanish
    for idx in propagated_indices:
        plot_concept_indices.add(idx)
        
    # Also add the top originally active concepts to provide context
    top_original_indices = base_concepts.argsort()[-top_k_concepts:][::-1]
    for idx in top_original_indices:
        plot_concept_indices.add(idx)
        
    final_plot_indices = sorted(list(plot_concept_indices), key=lambda i: base_concepts[i])
    concept_axis_names = [concept_names[i] for i in final_plot_indices]

    # ------------------------------------------------

    feature_tensor = torch.from_numpy(intervened[None, :]).to(device_obj)
    with torch.no_grad():
        logits = posthoc_layer.forward_projs(feature_tensor)
        post_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    base_pred = int(base_probs.argmax())
    post_pred = int(post_probs.argmax())

    class_union = sorted(
        set(base_probs.argsort()[-top_k_classes:]) | set(post_probs.argsort()[-top_k_classes:]),
        key=lambda idx: post_probs[idx],
    )[::-1]

    fig, axes = plt.subplots(1, 3, figsize=(20, 8))
    axes[0].imshow(raw_img)
    axes[0].axis("off")
    axes[0].set_title(f"Sample #{image_index}\nTrue: {class_names[label]}")

    y_labels = [class_names[i] for i in class_union][::-1]
    axes[1].barh(
        y_labels,
        base_probs[class_union][::-1],
        color="tab:gray",
        alpha=0.6,
        label="baseline",
    )
    axes[1].barh(
        y_labels,
        post_probs[class_union][::-1],
        color="tab:orange",
        alpha=0.7,
        label="after intervention",
    )
    axes[1].set_title("Class probabilities")
    axes[1].legend()

    # Plot Concepts
    y_pos = np.arange(len(concept_axis_names))
    axes[2].barh(
        y_pos,
        base_concepts[final_plot_indices],
        color="tab:blue",
        alpha=0.6,
        label="Before"
    )
    axes[2].barh(
        y_pos,
        intervened[final_plot_indices],
        color="tab:red",
        alpha=0.7,
        label="After"
    )
    axes[2].set_yticks(y_pos)
    axes[2].set_yticklabels(concept_axis_names)
    axes[2].set_title("Concept Activations (Showing Propagation)")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    print(f"Concepts implicitly zeroed: {[concept_names[i] for i in propagated_indices]}")

### Optional: perform_intervention_single (no propagation)
Use this helper when you want to clamp concept activations directly without triggering the entailment propagation mechanism.

In [ ]:
def perform_intervention_single(
    image_index: int,
    interventions: Dict[str, float],
    top_k_classes: int = 5,
) -> None:
    """Apply interventions by directly clamping concept activations (no propagation)."""

    if not interventions:
        raise ValueError("Provide at least one concept -> value entry.")

    raw_img, base_concepts, base_probs, label = compute_features_and_probs(image_index)

    intervened = base_concepts.copy()
    zeroed_concepts: List[str] = []
    applied_concepts: List[str] = []
    for concept_name, target_value in interventions.items():
        idx = concept_lookup.get(concept_name.lower())
        if idx is None:
            print(f"[warn] Concept '{concept_name}' not found; skipping.")
            continue
        intervened[idx] = float(target_value)
        applied_concepts.append(concept_names[idx])
        if abs(target_value) <= 1e-8:
            zeroed_concepts.append(concept_names[idx])

    if not applied_concepts:
        print("No valid interventions resolved; aborting.")
        return

    feature_tensor = torch.from_numpy(intervened[None, :]).to(device_obj)
    with torch.no_grad():
        logits = posthoc_layer.forward_projs(feature_tensor)
        post_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    base_pred = int(base_probs.argmax())
    post_pred = int(post_probs.argmax())

    class_union = sorted(
        set(base_probs.argsort()[-top_k_classes:]) | set(post_probs.argsort()[-top_k_classes:]),
        key=lambda idx: post_probs[idx],
    )[::-1]

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    axes[0].imshow(raw_img)
    axes[0].axis("off")
    axes[0].set_title(f"Sample #{image_index}\nTrue: {class_names[label]}")

    y_labels = [class_names[i] for i in class_union][::-1]
    axes[1].barh(
        y_labels,
        base_probs[class_union][::-1],
        color="tab:gray",
        alpha=0.6,
        label="baseline",
    )
    axes[1].barh(
        y_labels,
        post_probs[class_union][::-1],
        color="tab:orange",
        alpha=0.7,
        label="after intervention",
    )
    axes[1].set_title("Class probabilities (before vs after)")
    axes[1].set_xlabel("probability")
    axes[1].legend()

    axes[2].barh(
        applied_concepts[::-1],
        base_concepts[[concept_lookup[name.lower()] for name in applied_concepts]][::-1],
        color="tab:blue",
        alpha=0.6,
        label="baseline",
    )
    axes[2].barh(
        applied_concepts[::-1],
        intervened[[concept_lookup[name.lower()] for name in applied_concepts]][::-1],
        color="tab:red",
        alpha=0.7,
        label="intervened",
    )
    axes[2].set_title("Clamped concept activations")
    axes[2].set_xlabel("activation")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    print(
        f"Baseline prediction: {class_names[base_pred]} (p={base_probs[base_pred]:.3f}); "
        f"After intervention: {class_names[post_pred]} (p={post_probs[post_pred]:.3f})"
    )
    if zeroed_concepts:
        print("Concepts zeroed out:")
        for name in zeroed_concepts:
            print(f"  - {name}")
    deltas = [
        (class_names[idx], float(post_probs[idx] - base_probs[idx]))
        for idx in class_union
    ]
    print("Class probability deltas:")
    for name, delta in deltas:
        print(f"  {name:>15}: {delta:+.3f}")

## 7. Example usage
Pick an ImageNet validation index, inspect its baseline evidence with `analyze_image`, then craft a concept-to-value mapping passed into `perform_intervention` to observe the causal effect.

In [ ]:
print(artifact.keys())
print(artifact['train_samples_used'])

In [ ]:
MAX_CONCEPT_BARS = 10
sample_idx = 0

print(concept_bank.concept_names)

# enter intervention concept here
intervention_index = concept_bank.concept_names.index('a flag')

example_interventions = {
    concept_names[intervention_index]: 0.0
}

analyze_image(sample_idx)
perform_intervention(sample_idx, example_interventions, top_k_classes=6)
perform_intervention_single(sample_idx, example_interventions, top_k_classes=6)

In [ ]:
# Probability Response Curves: HypCBM vs PCBM (Additive)
# Metrics: Original Prediction vs. True Class Probability
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import random
from pathlib import Path
from types import SimpleNamespace

# --- Aliases ---
hyp_model = posthoc_layer
hyp_concepts = concept_tensor
hyp_backbone = backbone
hyp_device = device_obj

# --- PCBM State Management ---
_pcbm_state = None

def _ensure_pcbm_state():
    global _pcbm_state
    if _pcbm_state is not None:
        return _pcbm_state

    device = torch.device(DEVICE)
    # Update path if needed
    ckpt_path = Path("path/to/pcbm_checkpoint.ckpt").expanduser()
    
    args = SimpleNamespace(
        device=str(device),
        backbone_name="clip",
        hycoclip_config="hycoclip/configs/train_clip_vit_b.py",
        hycoclip_checkpoint="hycoclip/clip_vit_b.pth",
        out_dir="./data",
    )
    
    pcbm_backbone, pcbm_preprocess = get_model(args, backbone_name="clip")
    pcbm_backbone = pcbm_backbone.to(device).eval()

    obj = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(obj, PosthocLinearCBM):
        pcbm_posthoc = obj
    elif isinstance(obj, dict):
        pcbm_posthoc = None
        for key in ["model", "cbm", "posthoc_layer"]:
            maybe = obj.get(key)
            if isinstance(maybe, PosthocLinearCBM):
                pcbm_posthoc = maybe
                break
        if pcbm_posthoc is None:
            raise TypeError("Checkpoint does not contain a PosthocLinearCBM")
    else:
        raise TypeError("Unsupported checkpoint format for PCBM")

    concept_names = getattr(pcbm_posthoc, "names", [])
    concept_bank = ConceptBank(
        {
            name: (
                pcbm_posthoc.cavs[i].detach().cpu().numpy().reshape(1, -1),
                None,
                None,
                pcbm_posthoc.intercepts[i].detach().cpu().numpy().reshape(1, -1),
                {},
            )
            for i, name in enumerate(concept_names)
        },
        device,
    )
    
    pcbm_posthoc = pcbm_posthoc.to(device).eval()
    pcbm_concept_tensor = concept_bank.vectors.detach().clone().to(device)
    pcbm_concept_lookup = {name.lower(): idx for idx, name in enumerate(concept_names)}

    # Dataset
    data_root = Path("data/sun397/SUN_dataset/data").expanduser()
    full_dataset = datasets.SUN397(root=str(data_root), transform=pcbm_preprocess, download=False)
    val_size = max(1, int(len(full_dataset) * 0.1))
    train_size = len(full_dataset) - val_size
    generator = torch.Generator().manual_seed(0)
    _, val_subset = torch.utils.data.random_split(full_dataset, [train_size, val_size], generator=generator)
    
    base_raw = datasets.SUN397(root=str(data_root), transform=None, download=False)
    subset_indices = getattr(val_subset, "indices", list(range(len(val_subset))))
    raw_dataset = torch.utils.data.Subset(base_raw, subset_indices)
    class_names = full_dataset.classes

    def fetch_sample_pcbm(image_index: int):
        tensor_img, label = val_subset[image_index]
        raw_img, _ = raw_dataset[image_index]
        return tensor_img.unsqueeze(0).to(device), raw_img, label

    def compute_features_and_probs_pcbm(image_index: int):
        batch_tensor, raw_img, label = fetch_sample_pcbm(image_index)
        with torch.no_grad():
            embedding = pcbm_backbone.encode_image(batch_tensor, project=True).detach().cpu().numpy().astype(np.float32)
        features = pcbm_posthoc.compute_dist(torch.from_numpy(embedding).to(device)).cpu().numpy()
        feature_tensor = torch.from_numpy(features).to(device)
        with torch.no_grad():
            logits = pcbm_posthoc.forward_projs(feature_tensor)
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()[0]
        return raw_img, features[0].copy(), probs.copy(), np.asarray(label)

    def find_misclassified_indices_pcbm(max_mismatches: int = 500, log_every: int = 100):
        csv_path = Path("intervention_results_pcbm.csv")
        if not csv_path.exists(): return []
        df = pd.read_csv(csv_path)
        if "image_index" not in df.columns: return []
        return df["image_index"].dropna().astype(int).tolist()

    _pcbm_state = SimpleNamespace(
        device=device,
        backbone=pcbm_backbone,
        posthoc=pcbm_posthoc,
        concept_tensor=pcbm_concept_tensor,
        concept_names=concept_names,
        concept_lookup=pcbm_concept_lookup,
        class_names=class_names,
        fetch=fetch_sample_pcbm,
        compute=compute_features_and_probs_pcbm,
        find_misclassified=find_misclassified_indices_pcbm,
        val_subset=val_subset
    )
    return _pcbm_state

# --- Helpers ---
rng = np.random.default_rng(42)
from utils.hyperbolic_interventions import propagate_intervention, calculate_half_aperture, calculate_exterior_angle

def _choose_concept(contributions: np.ndarray, activations: np.ndarray, strategy: str) -> int:
    if strategy == "topk":
        return int(np.argmax(contributions))
    if strategy == "random":
        active = np.where(activations > 0)[0]
        pool = active if active.size else np.arange(len(activations))
        return int(rng.choice(pool))
    raise ValueError(f"Unknown strategy {strategy}")

def _scaled_propagation(intervened_np: np.ndarray, concept_idx: int, scale: float) -> np.ndarray:
    tensor = torch.from_numpy(intervened_np[None, :]).to(hyp_concepts.device, dtype=hyp_concepts.dtype)
    half_apertures = calculate_half_aperture(hyp_concepts, hyp_backbone).reshape(-1)
    parent_aperture = half_apertures[concept_idx]
    angles = calculate_exterior_angle(hyp_concepts, hyp_concepts[concept_idx].unsqueeze(0), hyp_backbone).reshape(-1)
    parent_norm = torch.norm(hyp_concepts[concept_idx], p=2)
    
    # Using +0.05 for more propagation, adjust as needed
    adaptive_eta = 8.62 * (parent_norm - 0.09) + 0.05
    
    threshold = adaptive_eta * parent_aperture
    child_mask = angles < threshold
    child_mask[concept_idx] = False
    child_mask = child_mask.to(tensor.device)
    tensor[:, child_mask] *= scale
    return tensor.cpu().numpy()[0]

def _sweep_hyp_case(image_index: int, concept_idx: int, deltas: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    _, feats, probs, label = compute_features_and_probs(image_index)
    pred_cls = int(np.argmax(probs))
    true_cls = int(label)
    
    pred_out = []
    true_out = []
    
    original_val = feats[concept_idx]
    
    for d in deltas:
        target_val = max(0.0, original_val - d)
        
        if original_val > 1e-6:
            effective_scale = target_val / original_val
        else:
            effective_scale = 0.0
            
        intervened = apply_interventions_to_features(
            feats[None, :],
            [{"idx": concept_idx, "value": float(target_val)}],
            hyp_concepts,
            model=hyp_backbone,
        )[0]

        if d > 0:
            intervened = _scaled_propagation(intervened, concept_idx, scale=effective_scale)

        feature_tensor = torch.from_numpy(intervened[None, :]).to(hyp_device)
        with torch.no_grad():
            logits = hyp_model.forward_projs(feature_tensor)
            post_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        
        pred_out.append(post_probs[pred_cls])
        true_out.append(post_probs[true_cls])
        
    return np.asarray(pred_out), np.asarray(true_out)

def _sweep_pcbm_case(image_index: int, concept_idx: int, deltas: np.ndarray, state=None) -> tuple[np.ndarray, np.ndarray]:
    state = _ensure_pcbm_state() if state is None else state
    _, feats, probs, label = state.compute(image_index)
    pred_cls = int(np.argmax(probs))
    true_cls = int(label)
    
    pred_out = []
    true_out = []
    
    original_val = feats[concept_idx]
    
    for d in deltas:
        perturbed = feats.copy()
    
        if original_val > 0:
            perturbed[concept_idx] = max(0.0, original_val - d)
        
        tensor = torch.from_numpy(perturbed[None, :]).to(state.device)
        with torch.no_grad():
            logits = state.posthoc.forward_projs(tensor)
            post_probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
        
        pred_out.append(post_probs[pred_cls])
        true_out.append(post_probs[true_cls])
        
    return np.asarray(pred_out), np.asarray(true_out)

# --- Builders ---
def _build_automated_cases(model_type: str, indices: list[int], limit: int = 100, state=None):
    cases = []
    for idx in indices[:limit]:
        if model_type == "hyp":
            _, feats, probs, _ = compute_features_and_probs(idx)
            weights = hyp_model.classifier.weight[int(np.argmax(probs))].detach().cpu().numpy()
        else:
            state = _ensure_pcbm_state() if state is None else state
            _, feats, probs, _ = state.compute(idx)
            weights = state.posthoc.classifier.weight[int(np.argmax(probs))].detach().cpu().numpy()
        contributions = weights * feats
        cases.append((idx, contributions, feats))
    return cases

def _cases_from_strategy(strategy: str, cases_raw: list[tuple], limit: int = 100):
    selected = []
    for idx, contribs, feats in cases_raw[:limit]:
        selected.append((idx, _choose_concept(contribs, feats, strategy)))
    return selected

def _load_manual_cases(csv_path: Path, limit: int = 100):
    df = pd.read_csv(csv_path)
    df = df.dropna(subset=["image_index", "chosen_concept_idx"])
    df = df.astype({"image_index": int, "chosen_concept_idx": int})
    subset = df[["image_index", "chosen_concept_idx"]].head(limit)
    return list(subset.itertuples(index=False, name=None))


def plot_probability_response_curves(limit: int = 100, deltas: np.ndarray | None = None):
    plt.style.use("seaborn-v0_8-paper") 
    plt.rcParams.update({
        "font.family": "serif",
        "font.size": 14,
        "axes.labelsize": 14,
        "axes.titlesize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "lines.markersize": 6,
        "lines.linewidth": 2.5
    })
    deltas = np.linspace(0, 0.6, 24) if deltas is None else np.asarray(deltas)
    
    pcbm_state = _ensure_pcbm_state()
    hyp_manual = _load_manual_cases(Path("intervention_results.csv"), limit)
    pcbm_manual = _load_manual_cases(Path("intervention_results_pcbm.csv"), limit)

    hyp_mis = list({case[0] for case in hyp_manual})
    pcbm_mis = list({case[0] for case in pcbm_manual})

    hyp_cases_raw = _build_automated_cases("hyp", hyp_mis, limit=limit)
    pcbm_cases_raw = _build_automated_cases("pcbm", pcbm_mis, limit=limit, state=pcbm_state)

    # --- 2. Define Strategies ---
    strategies = {
        "Manual Selection": (hyp_manual, pcbm_manual),
        "Top Contributing": (
            _cases_from_strategy("topk", hyp_cases_raw, limit),
            _cases_from_strategy("topk", pcbm_cases_raw, limit),
        ),
        "Random Selection": (
            _cases_from_strategy("random", hyp_cases_raw, limit),
            _cases_from_strategy("random", pcbm_cases_raw, limit),
        )
    }

    results = {"hyp": {}, "pcbm": {}}
    
    # --- 3. Create Subplots (1 Row, 3 Cols) ---
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
    strategy_names = ["Top Contributing"]

    for i, name in enumerate(strategy_names):
        print(f"--- Processing Strategy: {name} ---")
        
        # Filter for Common Images
        hyp_data = strategies[name][0]
        pcbm_data = strategies[name][1]
        
        hyp_img_ids = {c[0] for c in hyp_data}
        pcbm_img_ids = {c[0] for c in pcbm_data}
        common_ids = hyp_img_ids.intersection(pcbm_img_ids)
        
        if len(common_ids) == 0:
            print(f"[Warn] No common images for {name}. Skipping.")
            continue
            
        hyp_cases = [c for c in hyp_data if c[0] in common_ids]
        pcbm_cases = [c for c in pcbm_data if c[0] in common_ids]
        
        hyp_cases.sort(key=lambda x: x[0])
        pcbm_cases.sort(key=lambda x: x[0])
        
        # Run Sweep
        hyp_results = [_sweep_hyp_case(idx, cidx, deltas) for idx, cidx in hyp_cases]
        pcbm_results = [_sweep_pcbm_case(idx, cidx, deltas, pcbm_state) for idx, cidx in pcbm_cases]
        
        hyp_pred = np.stack([r[0] for r in hyp_results])
        pcbm_pred = np.stack([r[0] for r in pcbm_results])
        
        n_hyp = max(1, hyp_pred.shape[0])
        n_pcbm = max(1, pcbm_pred.shape[0])
        
        # Calculate Stats
        hm_p, hs_p = hyp_pred.mean(axis=0), hyp_pred.std(axis=0)/np.sqrt(n_hyp)
        pm_p, ps_p = pcbm_pred.mean(axis=0), pcbm_pred.std(axis=0)/np.sqrt(n_pcbm)

        # Plot
        ax = axes[i]
        ax.set_xlim(0,0.52)
        # HypCBM (Red)
        ax.plot(deltas, hm_p, "o-", color="#d62728", label="HypCBM")
        ax.fill_between(deltas, hm_p - hs_p, hm_p + hs_p, color="#d62728", alpha=0.2)
        
        # PCBM (Blue)
        ax.plot(deltas, pm_p, "d--", color="#1f77b4", label="PCBM")
        ax.fill_between(deltas, pm_p - ps_p, pm_p + ps_p, color="#1f77b4", alpha=0.15)
        
        # Styling
        ax.set_title(f"{name}\n(N={len(common_ids)})")
        ax.set_xlabel(r"Intervention Strength ($\Delta$)")
        if i == 0:
            ax.set_ylabel("Original Class Probability")
        
        ax.grid(True, alpha=0.3, linestyle='--')
        
        # Legend only in the middle plot (or first) to avoid clutter
        if i == 1:
            ax.legend(loc="upper right", frameon=True, fancybox=False, edgecolor='black')
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig('intervention_all_strategies.pdf', format='pdf', bbox_inches='tight')
    plt.show()

    return results, deltas

results, deltas = plot_probability_response_curves(limit=100)